# 🏍️ Frota de Motocicletas no Brasil — Janeiro 2020–2025 + Previsão

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model import LinearRegression

# Carrega o CSV unificado
df = pd.read_csv('senatran_csvs/frota_janeiro_2020_2025.csv')
df.head()

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Agrega total de motos por ano (soma todos os municípios)
df_anual = (
    df.groupby('ANO')['MOTOCICLETA']
    .sum()
    .reset_index()
    .rename(columns={'MOTOCICLETA': 'TOTAL_MOTOS'})
)

df_anual

In [ ]:
# Regressão linear para previsão 2026 e 2027
X = df_anual['ANO'].values.reshape(-1, 1)
y = df_anual['TOTAL_MOTOS'].values

modelo = LinearRegression()
modelo.fit(X, y)

anos_previsao = np.array([[2026], [2027]])
previsoes = modelo.predict(anos_previsao)

df_previsao = pd.DataFrame({
    'ANO': [2026, 2027],
    'TOTAL_MOTOS': previsoes,
    'TIPO': 'Previsão'
})

df_anual['TIPO'] = 'Real'

df_plot = pd.concat([df_anual, df_previsao], ignore_index=True)
df_plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

reais    = df_plot[df_plot['TIPO'] == 'Real']
previsao = df_plot[df_plot['TIPO'] == 'Previsão']

# Linha real
ax.plot(reais['ANO'], reais['TOTAL_MOTOS'],
        color='#E63946', linewidth=2.5, marker='o', markersize=7, label='Real')

# Linha de previsão (tracejada) conectada ao último ponto real
ultimo_real = reais.iloc[-1]
ax.plot(
    [ultimo_real['ANO']] + list(previsao['ANO']),
    [ultimo_real['TOTAL_MOTOS']] + list(previsao['TOTAL_MOTOS']),
    color='#E63946', linewidth=2.5, linestyle='--', marker='o',
    markersize=7, markerfacecolor='white', markeredgewidth=2,
    label='Previsão'
)

# Anotações nos pontos
for _, row in df_plot.iterrows():
    ax.annotate(
        f"{row['TOTAL_MOTOS']/1e6:.1f}M",
        xy=(row['ANO'], row['TOTAL_MOTOS']),
        xytext=(0, 12), textcoords='offset points',
        ha='center', fontsize=9,
        color='gray' if row['TIPO'] == 'Previsão' else '#333333'
    )

# Área sombreada de previsão
ax.axvspan(2025.5, 2027.5, alpha=0.06, color='gray', label='Período previsto')

ax.set_title('Frota de Motocicletas no Brasil\n(Janeiro de cada ano)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Ano', fontsize=11)
ax.set_ylabel('Total de Motocicletas', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.set_xticks(df_plot['ANO'])
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('senatran_csvs/frota_motos_previsao.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo em senatran_csvs/frota_motos_previsao.png')